In [ ]:
import os
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from scipy.stats import gaussian_kde
from scipy.signal import argrelextrema

# ====================================================================
# 1. CATALUMINESCENCE DATA LOADER AND STATISTICAL SUMMARY
# ====================================================================
def load_and_summarize_csv(file_path):
    """Load cataluminescence CSV from Agilent DAQ970A.
    
    These files have an instrument metadata header block (Address, Model,
    Serial Number, etc.) that must be skipped. The actual data header
    starts at the row containing 'Scan Num' or '101 ('.
    """
    header_idx = 0
    with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
        for idx, line in enumerate(f):
            if any(key in line for key in ['Scan Num', '101 (', 'Scan Swee']):
                header_idx = idx
                break
                
    df = pd.read_csv(file_path, skiprows=header_idx)
    df.columns = df.columns.str.strip()
    df = df.dropna(how='all', axis=1).dropna(how='all', axis=0)
    
    # Identify time column
    time_cols = [col for col in df.columns if 'time' in col.lower() or 'swee' in col.lower()]
    df['Timestamp'] = df[time_cols[0]] if time_cols else df.index
    
    # Identify sensor columns (contain 'C' for temperature channels)
    sensor_cols = [col for col in df.columns if 'C' in col]
    sensor_cols = [c for c in sensor_cols if c != 'Timestamp']
    
    for col in sensor_cols:
        df[col] = pd.to_numeric(df[col], errors='coerce')
        # Scrub hardware overloads (Agilent returns +9.9E+37 for overloads)
        overload_mask = (df[col].abs() > 10000) | (df[col] == np.inf)
        df.loc[overload_mask, col] = np.nan
        
    df[sensor_cols] = df[sensor_cols].ffill().bfill()
    
    print("\n==================================================")
    print(" RAW DATA POINT STATISTICS")
    print("==================================================")
    print(f"Total Rows Processed : {len(df)}")
    print(f"Total Sensor Channels: {len(sensor_cols)}")
    for col in sensor_cols:
        print(f"Channel {col} Mean: {df[col].mean():.2f} | Min: {df[col].min():.2f} | Max: {df[col].max():.2f}")
    print("==================================================\n")
        
    return df, sensor_cols

# ====================================================================
# 2. SEQUENCE GENERATOR
# ====================================================================
def create_sequences(data_array, timestamps, time_steps):
    Xs, ts = [], []
    for i in range(len(data_array) - time_steps):
        Xs.append(data_array[i:(i + time_steps)])
        ts.append(timestamps.iloc[i + time_steps - 1])
    return np.array(Xs), np.array(ts)

# ====================================================================
# 3. PYTORCH ARCHITECTURE
# ====================================================================
class LSTMAutoencoder(nn.Module):
    def __init__(self, num_features, hidden_size=64):
        super(LSTMAutoencoder, self).__init__()
        self.hidden_size = hidden_size
        
        self.encoder_lstm = nn.LSTM(
            input_size=num_features, hidden_size=hidden_size, batch_first=True
        )
        self.decoder_lstm = nn.LSTM(
            input_size=hidden_size, hidden_size=hidden_size, batch_first=True
        )
        self.output_layer = nn.Linear(hidden_size, num_features)
        
    def forward(self, x):
        batch_size, seq_len, _ = x.size()
        _, (hidden_state, _) = self.encoder_lstm(x)
        last_hidden_state = hidden_state[-1]
        repeated_hidden = last_hidden_state.unsqueeze(1).repeat(1, seq_len, 1)
        decoded, _ = self.decoder_lstm(repeated_hidden)
        reconstructed = self.output_layer(decoded)
        return reconstructed

# ====================================================================
# 4. TRAINING ENGINE WITH COSINE ANNEALING SCHEDULER
# ====================================================================
def train_model(model, train_loader, num_epochs=35, initial_lr=0.002):
    criterion = nn.L1Loss()
    optimizer = optim.Adam(model.parameters(), lr=initial_lr)
    
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs, eta_min=0.0001)
    
    model.train()
    for epoch in range(num_epochs):
        epoch_loss = 0.0
        for batch_x in train_loader:
            optimizer.zero_grad()
            outputs = model(batch_x[0])
            loss = criterion(outputs, batch_x[0])
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
            
        scheduler.step()
        
    return model

# ====================================================================
# 5. KDE THRESHOLDING AND METRICS EVALUATION
# ====================================================================
def evaluate_with_kde(model, X_seq_tensor, df_timestamps, df_original):
    model.eval()
    with torch.no_grad():
        reconstructed = model(X_seq_tensor)
        
    X_seq_np = X_seq_tensor.numpy()
    reconstructed_np = reconstructed.numpy()
    
    mae_loss = np.mean(np.abs(reconstructed_np - X_seq_np), axis=(1, 2))
    
    kde = gaussian_kde(mae_loss)
    x_range = np.linspace(min(mae_loss), max(mae_loss), 1000)
    kde_values = kde(x_range)
    
    minima_indices = argrelextrema(kde_values, np.less)[0]
    
    if len(minima_indices) > 0:
        threshold = x_range[minima_indices[0]]
        method_used = "KDE Local Minimum (Valley)"
    else:
        threshold = np.percentile(mae_loss, 97)
        method_used = "97th Percentile Fallback"
        
    max_mae = np.max(mae_loss)
    
    results_df = pd.DataFrame({
        'Timestamp': df_timestamps,
        'MAE_Score': mae_loss,
        'Is_Anomaly': mae_loss > threshold
    })
    
    def calculate_confidence(mae):
        if mae <= threshold:
            return 0.0
        if max_mae == threshold:
            return 100.0
        scaled = 50.0 + ((mae - threshold) / (max_mae - threshold)) * 50.0
        return min(100.0, scaled)
        
    results_df['Confidence_Score'] = results_df['MAE_Score'].apply(calculate_confidence)
    merged_df = pd.merge(results_df, df_original, on='Timestamp', how='inner')
    
    print("==================================================")
    print(" OPTIMIZED RECONSTRUCTION MAE STATISTICS")
    print("==================================================")
    print(f"Mean Error Score     : {np.mean(mae_loss):.4f}")
    print(f"Median Error Score   : {np.median(mae_loss):.4f}")
    print(f"Max Error Score      : {max_mae:.4f}")
    print(f"Threshold Method     : {method_used}")
    print(f"Calculated Threshold : {threshold:.4f}")
    print(f"Total Anomalies      : {results_df['Is_Anomaly'].sum()}")
    print("==================================================\n")
    
    return merged_df, threshold, x_range, kde_values

# ====================================================================
# 6. VISUALIZATION ENGINE
# ====================================================================
def plot_kde_distribution(x_range, kde_values, threshold):
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=x_range, y=kde_values, mode='lines', fill='tozeroy', name='Error Density', line=dict(color='blue')))
    
    fig.add_vline(x=threshold, line_dash="dash", line_color="red", annotation_text=f"Threshold: {threshold:.4f}")
    
    fig.update_layout(
        title="Kernel Density Estimation of Reconstruction Errors (Cataluminescence)",
        xaxis_title="Mean Absolute Error",
        yaxis_title="Probability Density",
        width=1000, height=500, template="plotly_white"
    )
    fig.show()

def plot_timeline_results(merged_df, sensor_cols, threshold):
    anomalies = merged_df[merged_df['Is_Anomaly'] == True]
    num_sensors = len(sensor_cols)
    
    fig = make_subplots(
        rows=num_sensors + 1, cols=1, shared_xaxes=True, vertical_spacing=0.04,
        subplot_titles=["<b>Reconstruction Error (MAE)</b>"] + [f"Channel: {col}" for col in sensor_cols]
    )
    
    normal_points = merged_df[merged_df['Is_Anomaly'] == False]
    
    fig.add_trace(go.Scatter(x=normal_points['Timestamp'], y=normal_points['MAE_Score'], mode='markers', marker=dict(color='blue', size=4, opacity=0.6), name='Normal MAE'), row=1, col=1)
    fig.add_trace(go.Scatter(x=anomalies['Timestamp'], y=anomalies['MAE_Score'], mode='markers', marker=dict(color='red', size=6, symbol='x'), name='Anomalous MAE'), row=1, col=1)
    
    fig.add_hline(y=threshold, line_dash="dash", line_color="black", row=1, col=1)
    
    colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#9467bd', '#8c564b']
    
    for i, col in enumerate(sensor_cols):
        row_idx = i + 2
        fig.add_trace(go.Scatter(x=merged_df['Timestamp'], y=merged_df[col], mode='lines', line=dict(color=colors[i % len(colors)], width=1.5), showlegend=False), row=row_idx, col=1)
        
        hover_template = "Time: %{x}<br>Value: %{y:.2f}<br>Confidence: %{customdata:.1f}%<extra></extra>"
        fig.add_trace(go.Scatter(x=anomalies['Timestamp'], y=anomalies[col], mode='markers', customdata=anomalies['Confidence_Score'], marker=dict(color='red', size=8, symbol='diamond'), hovertemplate=hover_template, showlegend=False), row=row_idx, col=1)
        fig.update_yaxes(title_text="Temp C", row=row_idx, col=1)

    fig.update_layout(title="Cataluminescence DAQ970A — Time Series Anomaly Overlay", height=200 + (180 * num_sensors), width=1250, hovermode='x unified')
    fig.show()

# ====================================================================
# 7. EXECUTION AND MODEL EXPORT
# ====================================================================
DATA_DIR = os.path.join("data", "cataluminiscence")
TIME_STEPS = 10

# Model weights save directory
weights_dir = "Saved_models"
os.makedirs(weights_dir, exist_ok=True)
WEIGHTS_FILE = os.path.join(weights_dir, "cataluminiscence_model_weight.pth")

# Find all CSV files in the cataluminescence data directory
csv_files = sorted([f for f in os.listdir(DATA_DIR) if f.endswith('.csv')])
print(f"Found {len(csv_files)} CSV files in {DATA_DIR}:")
for f in csv_files:
    print(f"  - {f}")

# Use the first available CSV for training
file_path = os.path.join(DATA_DIR, csv_files[0])
print(f"\nUsing: {file_path}")

if os.path.exists(file_path):
    df, sensors = load_and_summarize_csv(file_path)
    num_features = len(sensors)
    
    scaler = StandardScaler()
    df_scaled = pd.DataFrame(scaler.fit_transform(df[sensors]), columns=sensors)
    
    X_seq, t_seq = create_sequences(df_scaled, df['Timestamp'], TIME_STEPS)
    X_tensor = torch.tensor(X_seq, dtype=torch.float32)
    
    loader = DataLoader(TensorDataset(X_tensor), batch_size=16, shuffle=False)
    
    model = LSTMAutoencoder(num_features=num_features, hidden_size=64)
    
    print("==================================================")
    print(" MODEL ARCHITECTURE INITIALIZATION")
    print("==================================================")
    print(model)
    print("==================================================\n")
    
    print("Training optimized model with Cosine Annealing Scheduler...")
    trained_model = train_model(model, loader, num_epochs=35, initial_lr=0.002)
    
    # Save model weights
    torch.save(trained_model.state_dict(), WEIGHTS_FILE)
    print(f"Model weights successfully saved to: {WEIGHTS_FILE}")
    
    merged_results, decision_threshold, kde_x, kde_y = evaluate_with_kde(
        trained_model, X_tensor, t_seq, df
    )
    
    plot_kde_distribution(kde_x, kde_y, decision_threshold)
    plot_timeline_results(merged_results, sensors, decision_threshold)
else:
    print(f"File not found: {file_path}")

Found 3 CSV files in data\cataluminiscence:
  - 1781339209_2 2025-11-21 0.csv
  - 1781339239_2 2025-11-21 0.csv
  - 20260613T095926_21-11-2025 0 2025-11-21_cleaned.csv

Using: data\cataluminiscence\1781339209_2 2025-11-21 0.csv

 RAW DATA POINT STATISTICS
Total Rows Processed : 2770
Total Sensor Channels: 1
Channel 101 (°C) Mean: 297.91 | Min: 23.40 | Max: 427.73

 MODEL ARCHITECTURE INITIALIZATION
LSTMAutoencoder(
  (encoder_lstm): LSTM(1, 64, batch_first=True)
  (decoder_lstm): LSTM(64, 64, batch_first=True)
  (output_layer): Linear(in_features=64, out_features=1, bias=True)
)

Training optimized model with Cosine Annealing Scheduler...
Model weights successfully saved to: Saved_models\cataluminiscence_model_weight.pth
 OPTIMIZED RECONSTRUCTION MAE STATISTICS
Mean Error Score     : 0.0243
Median Error Score   : 0.0249
Max Error Score      : 0.0600
Threshold Method     : KDE Local Minimum (Valley)
Calculated Threshold : 0.0177
Total Anomalies      : 1744

